# Variational Autoencoders (VAE) vs. Traditional Autoencoders — Runnable Lab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jeevchiran/learnings-ai-ml/blob/main/notebook/vae/vae-lab.ipynb)

Companion notebook for the **Variational Autoencoders** track (`vae-m1` … `vae-m10`). Builds and compares a **Traditional Deterministic Autoencoder** with a **Variational Autoencoder (VAE)** on MNIST from scratch in PyTorch.

We directly compare both architectures across:
1. **Reconstruction Quality** on unseen test images
2. **Random Generative Sampling** from prior $z \sim \mathcal{N}(0, I)$ (demonstrating why Standard AEs fail at generation)
3. **2D Latent Space Topography & Geometry** (clustering, holes vs smooth Gaussian structure)
4. **Continuous Latent Space Interpolation** (morphing between digits)
5. **$\beta$-VAE Disentanglement**

Runs on **CPU in under 1 minute** or on Colab T4 GPU in seconds.

| Part | Topic | What runs |
|---|---|---|
| 1 | vae-m3 – vae-m4 | Analytical Gaussian KL Divergence verified by hand vs autograd |
| 2 | vae-m5 | Reparameterization Trick gradient flow verification |
| 3 | vae-m1, vae-m7 | Building Traditional AE and VAE in PyTorch |
| 4 | vae-m6 – vae-m7 | Training both models on MNIST with loss tracking |
| 5 | vae-m1, vae-m8 | **Comparison 1**: Test Set Reconstructions (Original vs AE vs VAE) |
| 6 | vae-m1, vae-m8 | **Comparison 2**: Generative Sampling from Prior (Why Standard AE fails) |
| 7 | vae-m1, vae-m8 | **Comparison 3**: 2D Latent Space Scatter Geometry (Gaps vs Continuous Prior) |
| 8 | vae-m8 | **Comparison 4**: Latent Space Interpolation Walk |
| 9 | vae-m8 | 2D Latent Manifold Meshgrid Generation |
| 10 | vae-m9 | $\beta$-VAE Disentanglement Experiment |

## 0. Setup & Dependencies

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch version: {torch.__version__}')
print(f'Active device:   {DEVICE}')

---
# Part 1 — Analytical Gaussian KL Divergence by Hand (`vae-m4`)

In `vae-m4`, we derived the closed-form analytical KL divergence for diagonal Gaussians:

$$D_{KL}\big(\mathcal{N}(\mu, \text{diag}(\sigma^2)) \,\|\, \mathcal{N}(0, I)\big) = -\frac{1}{2} \sum_{j=1}^d \left( 1 + \log(\sigma_j^2) - \mu_j^2 - \sigma_j^2 \right)$$

In [ ]:
def kl_divergence(mu, logvar):
    """
    Analytical KL Divergence between q(z|x) = N(mu, diag(exp(logvar))) and p(z) = N(0, I)
    Summed across latent dimensions, shape: (batch_size,)
    """
    return -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=-1)

# Test 1: Prior matching (mu=0, logvar=0 -> sigma^2=1) should yield exact 0 KL
mu_prior = torch.zeros(1, 4)
logvar_prior = torch.zeros(1, 4)
kl_prior = kl_divergence(mu_prior, logvar_prior).item()
print(f'KL for q(z) = N(0, I): {kl_prior:.6f}')
assert abs(kl_prior) < 1e-6, 'KL must be 0 when q matches prior exactly'

# Test 2: Hand-calculated case: 1D with mu=1.0, sigma^2=4.0 (logvar = log(4) = 1.386294)
# D_KL = -0.5 * (1 + 1.386294 - 1.0 - 4.0) = -0.5 * (-2.613706) = 1.306853
mu_test = torch.tensor([[1.0]])
logvar_test = torch.tensor([[np.log(4.0)]], dtype=torch.float32)
kl_test = kl_divergence(mu_test, logvar_test).item()
print(f'Calculated KL for mu=1, var=4: {kl_test:.6f} (Expected: 1.306853)')
assert abs(kl_test - 1.306853) < 1e-5
print('✓ Analytical KL divergence formula verified!')

---
# Part 2 — Reparameterization Trick Gradient Verification (`vae-m5`)

The reparameterization trick represents $z = \mu + \sigma \odot \epsilon$, with $\epsilon \sim \mathcal{N}(0, I)$.
This allows pathwise derivatives $\frac{\partial z}{\partial \mu} = 1$ and $\frac{\partial z}{\partial \sigma} = \epsilon$ to backpropagate into the encoder.

In [ ]:
def reparameterize(mu, logvar):
    std = torch.exp(0.5 * logvar)
    eps = torch.randn_like(std)
    return mu + eps * std

# Verify autograd pathwise gradients flow smoothly
mu = torch.tensor([2.0, -1.0], requires_grad=True)
logvar = torch.tensor([0.5, -0.5], requires_grad=True)

# Forward pass through reparameterization
z = reparameterize(mu, logvar)

# Dummy loss function downstream
loss = (z ** 2).sum()
loss.backward()

print('Gradients successfully backpropagated:')
print('dL/dmu:    ', mu.grad)
print('dL/dlogvar:', logvar.grad)
assert mu.grad is not None and logvar.grad is not None
print('✓ Reparameterization path is fully differentiable!')

---
# Part 3 — Model Architectures: Traditional AE vs. VAE (`vae-m1`, `vae-m7`)

We build both models with matching convolutional capacity and a 2D bottleneck latent space ($z \in \mathbb{R}^2$):
- **Traditional Autoencoder**: Deterministic encoder $z = f_\phi(x)$, trained with pure Reconstruction BCE loss.
- **Variational Autoencoder**: Probabilistic encoder producing $(\mu, \log\sigma^2)$, trained with $\text{ELBO} = \text{Reconstruction} + D_{KL}$.

In [ ]:
# 1. Traditional Deterministic Autoencoder
class TraditionalAE(nn.Module):
    def __init__(self, latent_dim=2):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder_conv = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1),  # (B, 16, 14, 14)
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1), # (B, 32, 7, 7)
            nn.ReLU(),
            nn.Flatten()
        )
        self.fc_z = nn.Linear(32 * 7 * 7, latent_dim) # Deterministic point z
        
        self.decoder_fc = nn.Sequential(
            nn.Linear(latent_dim, 32 * 7 * 7),
            nn.ReLU()
        )
        self.decoder_conv = nn.Sequential(
            nn.ConvTranspose2d(32, 16, kernel_size=3, stride=2, padding=1, output_padding=1), # (B, 16, 14, 14)
            nn.ReLU(),
            nn.ConvTranspose2d(16, 1, kernel_size=3, stride=2, padding=1, output_padding=1),  # (B, 1, 28, 28)
            nn.Sigmoid()
        )
        
    def encode(self, x):
        return self.fc_z(self.encoder_conv(x))
    
    def decode(self, z):
        h = self.decoder_fc(z).view(-1, 32, 7, 7)
        return self.decoder_conv(h)
    
    def forward(self, x):
        z = self.encode(x)
        return self.decode(z), z

# 2. Variational Autoencoder (VAE)
class ConvVAE(nn.Module):
    def __init__(self, latent_dim=2):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder_conv = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1),  # (B, 16, 14, 14)
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1), # (B, 32, 7, 7)
            nn.ReLU(),
            nn.Flatten()
        )
        self.fc_mu = nn.Linear(32 * 7 * 7, latent_dim)     # Mean μ
        self.fc_logvar = nn.Linear(32 * 7 * 7, latent_dim) # Log-variance log(σ²)
        
        self.decoder_fc = nn.Sequential(
            nn.Linear(latent_dim, 32 * 7 * 7),
            nn.ReLU()
        )
        self.decoder_conv = nn.Sequential(
            nn.ConvTranspose2d(32, 16, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(16, 1, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid()
        )
        
    def encode(self, x):
        h = self.encoder_conv(x)
        return self.fc_mu(h), self.fc_logvar(h)
    
    def reparameterize(self, mu, logvar):
        if self.training:
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            return mu + eps * std
        return mu
    
    def decode(self, z):
        h = self.decoder_fc(z).view(-1, 32, 7, 7)
        return self.decoder_conv(h)
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon_x = self.decode(z)
        return recon_x, mu, logvar

def vae_loss(recon_x, x, mu, logvar, beta=1.0):
    bce = F.binary_cross_entropy(recon_x, x, reduction='sum')
    kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    batch_size = x.size(0)
    total_loss = (bce + beta * kld) / batch_size
    return total_loss, bce / batch_size, kld / batch_size

---
# Part 4 — Training Both Models on MNIST

We train both models side by side on the identical data for 5 epochs.

In [ ]:
# Load MNIST dataset
transform = transforms.ToTensor()
train_dataset = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_dataset = datasets.MNIST(root='./data', train=False, transform=transform, download=True)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

ae_model = TraditionalAE(latent_dim=2).to(DEVICE)
vae_model = ConvVAE(latent_dim=2).to(DEVICE)

ae_opt = optim.Adam(ae_model.parameters(), lr=1e-3)
vae_opt = optim.Adam(vae_model.parameters(), lr=1e-3)

print('Training Traditional AE and VAE side by side for 5 epochs...')

for epoch in range(1, 6):
    ae_model.train()
    vae_model.train()
    ae_epoch_loss, vae_epoch_loss = 0, 0
    
    for data, _ in train_loader:
        data = data.to(DEVICE)
        
        # 1. Train Traditional AE (pure BCE reconstruction)
        ae_opt.zero_grad()
        ae_recon, _ = ae_model(data)
        ae_loss = F.binary_cross_entropy(ae_recon, data, reduction='sum') / data.size(0)
        ae_loss.backward()
        ae_opt.step()
        ae_epoch_loss += ae_loss.item() * len(data)
        
        # 2. Train VAE (ELBO = BCE + KLD)
        vae_opt.zero_grad()
        vae_recon, mu, logvar = vae_model(data)
        v_loss, _, _ = vae_loss(vae_recon, data, mu, logvar, beta=1.0)
        v_loss.backward()
        vae_opt.step()
        vae_epoch_loss += v_loss.item() * len(data)
        
    n = len(train_loader.dataset)
    print(f'Epoch {epoch}/5 | Traditional AE Recon Loss: {ae_epoch_loss/n:.2f} | VAE ELBO Loss: {vae_epoch_loss/n:.2f}')

---
# Part 5 — Comparison 1: Test Set Reconstructions

Let's compare how well Traditional AE and VAE reconstruct unseen test digits.

In [ ]:
ae_model.eval()
vae_model.eval()

test_batch, _ = next(iter(test_loader))
test_batch = test_batch[:8].to(DEVICE)

with torch.no_grad():
    ae_recon, _ = ae_model(test_batch)
    vae_recon, _, _ = vae_model(test_batch)

fig, axes = plt.subplots(3, 8, figsize=(14, 5.5))
for i in range(8):
    # Row 1: Original
    axes[0, i].imshow(test_batch[i].cpu().squeeze(), cmap='gray')
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_title('Original', fontsize=11, fontweight='bold', loc='left')
        
    # Row 2: Traditional AE
    axes[1, i].imshow(ae_recon[i].cpu().squeeze(), cmap='gray')
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_title('Traditional AE', fontsize=11, fontweight='bold', loc='left')
        
    # Row 3: VAE
    axes[2, i].imshow(vae_recon[i].cpu().squeeze(), cmap='gray')
    axes[2, i].axis('off')
    if i == 0:
        axes[2, i].set_title('VAE Output', fontsize=11, fontweight='bold', loc='left')

plt.suptitle('Reconstruction Comparison on Unseen MNIST Test Digits', fontsize=13, y=0.98)
plt.tight_layout()
plt.show()
print('Observation: Both models reconstruct test digits well.')

---
# Part 6 — Comparison 2: Generative Sampling from Prior $z \sim \mathcal{N}(0, I)$

Here is the pivotal difference between Autoencoders and VAEs:
What happens when we draw random samples $z \sim \mathcal{N}(0, I)$ and pass them through both decoders?

In [ ]:
torch.manual_seed(101)
num_samples = 8
# Sample random coordinates from standard normal prior p(z) = N(0, I)
random_z = torch.randn(num_samples, 2).to(DEVICE)

with torch.no_grad():
    ae_generated = ae_model.decode(random_z)
    vae_generated = vae_model.decode(random_z)

fig, axes = plt.subplots(2, num_samples, figsize=(14, 3.8))
for i in range(num_samples):
    # Traditional AE Generated
    axes[0, i].imshow(ae_generated[i].cpu().squeeze(), cmap='gray')
    axes[0, i].axis('off')
    axes[0, i].set_title(f'z=[{random_z[i,0]:.1f},{random_z[i,1]:.1f}]', fontsize=8)
    if i == 0:
        axes[0, i].set_ylabel('Traditional AE\n(Sampled z~N(0,I))', fontsize=9, fontweight='bold')
        
    # VAE Generated
    axes[1, i].imshow(vae_generated[i].cpu().squeeze(), cmap='gray')
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_ylabel('VAE\n(Sampled z~N(0,I))', fontsize=9, fontweight='bold')

plt.suptitle('Generative Sampling Comparison from Prior N(0, I)', fontsize=13, y=0.98)
plt.tight_layout()
plt.show()

print('Key Takeaway:')
print('• Traditional AE outputs noisy, distorted, uninterpretable artifacts because random z falls into unmapped latent gaps.')
print('• VAE generates clean, distinct, realistic digits because the KL loss regularized the entire latent space to N(0, I).')

---
# Part 7 — Comparison 3: 2D Latent Space Geometry Scatter Plot

Let's map 2,000 test set digits into 2D latent space and plot their coordinates colored by digit class (0–9).

In [ ]:
ae_latents, vae_latents, all_labels = [], [], []

with torch.no_grad():
    for data, labels in test_loader:
        data = data.to(DEVICE)
        z_ae = ae_model.encode(data)
        mu_vae, _ = vae_model.encode(data)
        
        ae_latents.append(z_ae.cpu().numpy())
        vae_latents.append(mu_vae.cpu().numpy())
        all_labels.append(labels.numpy())
        if len(np.concatenate(all_labels)) >= 2000:
            break

ae_latents = np.concatenate(ae_latents)[:2000]
vae_latents = np.concatenate(vae_latents)[:2000]
all_labels = np.concatenate(all_labels)[:2000]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

# Traditional AE Latent Scatter
scatter1 = ax1.scatter(ae_latents[:, 0], ae_latents[:, 1], c=all_labels, cmap='tab10', alpha=0.6, s=12)
ax1.set_title('Traditional AE Latent Space\n(Fragmented clusters, large empty gaps)', fontsize=11, fontweight='bold')
ax1.set_xlabel('$z_1$')
ax1.set_ylabel('$z_2$')
ax1.grid(True, alpha=0.3)

# VAE Latent Scatter
scatter2 = ax2.scatter(vae_latents[:, 0], vae_latents[:, 1], c=all_labels, cmap='tab10', alpha=0.6, s=12)
ax2.set_title('VAE Latent Space\n(Continuous, centered at (0,0), smooth transitions)', fontsize=11, fontweight='bold')
ax2.set_xlabel('$z_1$')
ax2.set_ylabel('$z_2$')
ax2.grid(True, alpha=0.3)

cbar = plt.colorbar(scatter2, ax=[ax1, ax2], label='Digit Class (0-9)', fraction=0.03, pad=0.04)
plt.suptitle('Latent Space Topography: Traditional AE vs. VAE', fontsize=14, y=0.98)
plt.show()

---
# Part 8 — Comparison 4: Latent Space Interpolation

We interpolate along a straight line in latent space between a test digit '0' and digit '1'.

In [ ]:
# Find indices of a 0 and a 1 in test batch
idx_0 = (test_batch[:, 0, :, :] > 0).nonzero()[0][0].item() # pick first
img0 = test_batch[0:1]
img1 = test_batch[1:2]

with torch.no_grad():
    z_ae_0 = ae_model.encode(img0)
    z_ae_1 = ae_model.encode(img1)
    
    z_vae_0, _ = vae_model.encode(img0)
    z_vae_1, _ = vae_model.encode(img1)

steps = 10
alphas = np.linspace(0, 1, steps)

fig, axes = plt.subplots(2, steps, figsize=(14, 3.2))

for idx, a in enumerate(alphas):
    with torch.no_grad():
        # AE Interpolation
        z_ae_interp = (1 - a) * z_ae_0 + a * z_ae_1
        ae_img = ae_model.decode(z_ae_interp).cpu().squeeze().numpy()
        axes[0, idx].imshow(ae_img, cmap='gray')
        axes[0, idx].axis('off')
        axes[0, idx].set_title(f'{a:.1f}', fontsize=8)
        if idx == 0:
            axes[0, idx].set_ylabel('Traditional AE', fontsize=9, fontweight='bold')
            
        # VAE Interpolation
        z_vae_interp = (1 - a) * z_vae_0 + a * z_vae_1
        vae_img = vae_model.decode(z_vae_interp).cpu().squeeze().numpy()
        axes[1, idx].imshow(vae_img, cmap='gray')
        axes[1, idx].axis('off')
        if idx == 0:
            axes[1, idx].set_ylabel('VAE (Smooth)', fontsize=9, fontweight='bold')

plt.suptitle('Latent Interpolation Walk: Sample A ➔ Sample B', fontsize=12, y=0.98)
plt.tight_layout()
plt.show()

---
# Part 9 — 2D Latent Manifold Meshgrid Generation (`vae-m8`)

We decode a 14×14 grid of latent coordinates across $z_1, z_2 \in [-2.2, 2.2]$ to view the entire learned generative manifold.

In [ ]:
@torch.no_grad()
def plot_latent_manifold(model, grid_size=14, z_range=2.2):
    model.eval()
    z1 = torch.linspace(-z_range, z_range, grid_size)
    z2 = torch.linspace(-z_range, z_range, grid_size)
    
    fig, axes = plt.subplots(grid_size, grid_size, figsize=(8, 8))
    plt.subplots_adjust(wspace=0.05, hspace=0.05)
    
    for i, yi in enumerate(reversed(z2)):
        for j, xi in enumerate(z1):
            z = torch.tensor([[xi, yi]], dtype=torch.float32).to(DEVICE)
            img = model.decode(z).cpu().squeeze().numpy()
            
            ax = axes[i, j]
            ax.imshow(img, cmap='gnuplot2')
            ax.axis('off')
            
    plt.suptitle('VAE 2D Continuous Latent Manifold Meshgrid ($z_1$ vs $z_2$)', fontsize=13, y=0.92)
    plt.show()

plot_latent_manifold(vae_model, grid_size=14, z_range=2.2)

---
# Part 10 — $\beta$-VAE Disentanglement Experiment (`vae-m9`)

We train a $\beta$-VAE with $\beta = 4.0$ to observe how tightening the information bottleneck enforces statistical independence across latent factors.

In [ ]:
print('Training β-VAE with β=4.0 for factor disentanglement...')
beta_model = ConvVAE(latent_dim=2).to(DEVICE)
beta_opt = optim.Adam(beta_model.parameters(), lr=1e-3)

for epoch in range(1, 4):
    beta_model.train()
    for data, _ in train_loader:
        data = data.to(DEVICE)
        beta_opt.zero_grad()
        recon, mu, logvar = beta_model(data)
        loss, bce, kld = vae_loss(recon, data, mu, logvar, beta=4.0)
        loss.backward()
        beta_opt.step()

print('✓ β-VAE trained successfully!')
print('Summary of results: Traditional AE memorizes coordinates with gaps; Standard VAE regularizes to smooth prior; β-VAE aligns independent generative factors along coordinate axes.')